In [0]:
%sql
use catalog classic_workspace;
use schema default;

In [0]:
%python
from pyspark.sql import Row
from datetime import datetime, timedelta
import random

# Build a small batch of event rows inline — no external data needed
base_time = datetime(2024, 1, 15, 9, 0, 0)
batch_1 = [
    Row(event_id=i,
        event_type=random.choice(["click", "view", "purchase"]),
        user_id=random.randint(100, 199),
        event_ts=base_time + timedelta(seconds=i * 30))
    for i in range(1, 11)
]

df_batch_1 = spark.createDataFrame(batch_1)
display(df_batch_1)

In [0]:
%python
df_batch_1.write \
    .format("delta") \
    .mode("append") \
    .saveAsTable("events")

print("Batch 1 written — 10 rows appended.")

In [0]:
%python
base_time_2 = datetime(2024, 1, 15, 14, 0, 0)
batch_2 = [
    Row(event_id=i,
        event_type=random.choice(["click", "view", "purchase"]),
        user_id=random.randint(200, 299),
        event_ts=base_time_2 + timedelta(seconds=i * 30))
    for i in range(11, 21)
]

df_batch_2 = spark.createDataFrame(batch_2)
df_batch_2.display()

In [0]:
%python
df_batch_2.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("events")

print("Batch 2 written — table overwritten with 10 new rows.")

In [0]:
describe history events;

In [0]:
select count(*) from events;

In [0]:
%python
location = spark.sql("DESCRIBE DETAIL events") \
                .select("location") \
                .collect()[0][0]

print(f"Delta table path: {location}")  

In [0]:
%python
df_path = spark.read.format("delta").load(location)
print(f"Reading from: {location}")
print(f"Row count: {df_path.count()}")
display(df_path)

In [0]:
%python
df_catalog = spark.table("events")
print(f"Row count: {df_catalog.count()}")
display(df_catalog)